In [ ]:
### train시 1시간 정도 소모 됩니다. colab환경에서 아래를 browser의 console 에서 붙여 넣기가 필요할 수 있습니다.
### shift+cntr+i 로 browser console 열기
# https://github.com/chulminkw/DLCV/blob/master/data/util/colab_autoclick.js
'''
function ClickConnect(){
console.log("Working");
document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect,60000)
'''

# step1. Colab 환경 설정 및 데이터 준비"

## 1-1. 라이브러리 설치 및 드라이브 마운트

In [ ]:
# 1. YOLOv8 라이브러리 설치
!pip install ultralytics -q

# 2. 구글 드라이브 마운트 (데이터셋을 드라이브에 올린 경우)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

In [ ]:

CLASS_NAMES = [
    "Front fender(L)", "Rear bumper", "Front Wheel(R)", "Trunk lid", "Rocker panel(L)",
    "Front fender(R)", "Front bumper", "Bonnet", "Rear Wheel(R)", "Rear door(R)",
    "Front door(R)", "Head lights(R)", "Rear fender(R)", "Rear fender(L)", "Rocker panel(R)",
    "Rear lamp(L)", "Side mirror(R)", "Rear Wheel(L)", "Rear door(L)", "Side mirror(L)",
    "Head lights(L)", "Front Wheel(L)", "Front door(L)", "Rear lamp(R)", "Windshield",
    "Roof", "Undercarriage", "Rear windshield", "C pillar(L)", "A pillar(L)",
    "C pillar(R)", "A pillar(R)"
]

# ==========================================
# 2. 8면 분류 로직 (Tie-Breaking 강화)
# ==========================================
def classify_view_logic(yolo_results):
    detected = []
    boxes = yolo_results[0].boxes
    if boxes is None: return "Unknown", {'F':0, 'B':0, 'L':0, 'R':0}

    for box in boxes:
        if box.conf > 0.25:
            cls_id = int(box.cls)
            detected.append(CLASS_NAMES[cls_id])

    scores = {'F': 0, 'B': 0, 'L': 0, 'R': 0}

    # -----------------------------------------------------------
    # [Step 1] 점수 부여
    # -----------------------------------------------------------
    for cls in detected:
        # Front
        if cls == 'Front bumper': scores['F'] += 4
        elif cls == 'Bonnet': scores['F'] += 3
        elif cls == 'Windshield': scores['F'] += 1
        elif 'Head lights' in cls: scores['F'] += 2

        # Back
        if cls == 'Rear bumper': scores['B'] += 4
        elif cls == 'Trunk lid': scores['B'] += 3
        elif cls == 'Rear windshield': scores['B'] += 1
        elif 'Rear lamp' in cls: scores['B'] += 2

        # Side (L/R)
        if '(L)' in cls: scores['L'] += 3
        if '(R)' in cls: scores['R'] += 3

        # Corner Parts (Fender) -> 양면성 부여
        if 'Front fender' in cls: scores['F'] += 2
        if 'Rear fender' in cls: scores['B'] += 2

    # -----------------------------------------------------------
    # [Step 2] 뷰 결정 로직 (수정됨)
    # -----------------------------------------------------------
    F, B, L, R = scores['F'], scores['B'], scores['L'], scores['R']

    view = "Unknown"

    # 1. 양안 시각 (빼박 정면/후면)
    has_L_lamp = any('Head lights(L)' in d for d in detected)
    has_R_lamp = any('Head lights(R)' in d for d in detected)
    is_pure_front = has_L_lamp and has_R_lamp

    has_L_rear = any('Rear lamp(L)' in d for d in detected)
    has_R_rear = any('Rear lamp(R)' in d for d in detected)
    is_pure_back = has_L_rear and has_R_rear

    # 2. 판단 트리
    if is_pure_front:
        view = "Front"
    elif is_pure_back:
        view = "Back"

    # [수정 포인트 1] 정면/후면 판단 기준 완화 (동점 처리)
    # F가 어느정도 있고(>=2), L과 R의 차이가 크지 않으면(<=2) -> 정면으로 간주
    # 예: F[2], L[3], R[3] -> Front
    elif F >= 2 and abs(L - R) <= 2: view = "Front"
    elif B >= 2 and abs(L - R) <= 2: view = "Back"

    # [수정 포인트 2] 대각선 판단 시 '반대쪽보다 커야 한다' 조건 추가 (L > R)
    # F[2], L[3], R[3] 인 경우 -> L > R (False) 이므로 아래 조건 통과 못함 -> 오분류 방지
    elif F >= 2 and L >= 2 and L > R: view = "Front-Left"
    elif F >= 2 and R >= 2 and R > L: view = "Front-Right"
    elif B >= 2 and L >= 2 and L > R: view = "Back-Left"
    elif B >= 2 and R >= 2 and R > L: view = "Back-Right"

    # 완전 측면
    elif L > R and L > F and L > B: view = "Left"
    elif R > L and R > F and R > B: view = "Right"

    # Fallback
    else:
        # 점수가 너무 낮거나 애매한 경우
        max_score = max(F, B, L, R)
        if max_score == 0: view = "Unknown"
        # 최후의 수단: 단순히 점수 제일 높은 것 (동점이면 Front 우선)
        elif F == max_score: view = "Front"
        elif B == max_score: view = "Back"
        elif L == max_score: view = "Left"
        elif R == max_score: view = "Right"

    return view, scores

# ==========================================
# 3. 시각화 함수
# ==========================================
def visualize_result(img, results, view_pred, scores):
    img_vis = img.copy()
    annotator = results[0].plot()
    img_vis = annotator
    h, w = img_vis.shape[:2]

    cv2.rectangle(img_vis, (0, 0), (w, 50), (0, 0, 0), -1)
    cv2.rectangle(img_vis, (0, h-40), (w, h), (0, 0, 0), -1)

    # View Result
    cv2.putText(img_vis, f"VIEW: {view_pred}", (15, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA)

    # Score Details
    cv2.putText(img_vis, f"Scores: F[{scores['F']}] B[{scores['B']}] L[{scores['L']}] R[{scores['R']}]", (15, h-12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1, cv2.LINE_AA)

    return img_vis

# ==========================================
# 4. 배치 실행 함수
# ==========================================
def run_batch_view_classification(start_idx=0, batch_size=10):
    model = YOLO(MODEL_PATH)

    img_files = sorted(glob.glob(os.path.join(TEST_IMG_DIR, "*.*")))

    if start_idx >= len(img_files):
        print(f"❌ Error: Start index {start_idx} exceeds total images")
        return

    end_idx = min(start_idx + batch_size, len(img_files))
    batch_files = img_files[start_idx : end_idx]

    print(f"🚀 Processing {len(batch_files)} images (Index: {start_idx} ~ {end_idx-1})...")

    plt.figure(figsize=(12, 6 * len(batch_files)))

    for i, file_path in enumerate(batch_files):
        img = cv2.imread(file_path)
        if img is None: continue

        results = model.predict(img, conf=0.25, verbose=False)
        view_pred, scores = classify_view_logic(results)
        final_img = visualize_result(img, results, view_pred, scores)

        final_img_rgb = cv2.cvtColor(final_img, cv2.COLOR_BGR2RGB)

        plt.subplot(len(batch_files), 1, i + 1)
        plt.imshow(final_img_rgb)
        plt.axis('off')
        plt.title(f"File: {os.path.basename(file_path)}", fontsize=10)

    plt.tight_layout()
    plt.show()

In [ ]:
TEST_IMG_DIR = "/content/drive/MyDrive/03. HDMF/(share)HDMF_AUTO_SPOKE/DATA/04_DATA/balanced_dataset_split_polygon/test/images"
MODEL_PATH = "/content/drive/MyDrive/03. HDMF/(pre_study)2026_HDMF_AUTO_SPOKE/SUBJECT/WEEK4_CAR_DAMAGE_SEGMENTATION/FINE_TUNING_MODEL/yolov8x_car_parts_seg/weights/best.pt"

In [ ]:
if __name__ == "__main__":
    # 0번부터 10장 확인
    run_batch_view_classification(start_idx=0, batch_size=10)

In [ ]:
if __name__ == "__main__":
    # 0번부터 10장 확인
    run_batch_view_classification(start_idx=10, batch_size=10)

In [ ]:
if __name__ == "__main__":
    # 0번부터 10장 확인
    run_batch_view_classification(start_idx=20, batch_size=10)

In [ ]:
# @title
if __name__ == "__main__":
    # 0번부터 10장 확인
    run_batch_view_classification(start_idx=30, batch_size=10)

In [ ]:
if __name__ == "__main__":
    # 0번부터 10장 확인
    run_batch_view_classification(start_idx=40, batch_size=10)

In [ ]:
if __name__ == "__main__":
    # 0번부터 10장 확인
    run_batch_view_classification(start_idx=50, batch_size=10)

In [ ]:
if __name__ == "__main__":
    # 0번부터 10장 확인
    run_batch_view_classification(start_idx=60, batch_size=10)

In [ ]:
if __name__ == "__main__":
    # 0번부터 10장 확인
    run_batch_view_classification(start_idx=70, batch_size=10)